# 🎬 AutoTube AI — 1-Click Mobile Cloud Studio
### 100% Free Cloud Execution • Runs on Google Cloud (Free T4 GPU/CPU) • No Laptop Needed

Welcome to **AutoTube AI Mobile Cloud Studio**! This notebook is fully optimized to run on **mobile phones (Safari / Chrome)** as well as laptops.

---
### 📱 How to Use in 2 Simple Taps:
1. **Tap ▶️ on Step 1 below**: Installs FFmpeg and AutoTube AI engine (~1 min).
2. **Tap ▶️ on Step 2 (Production Studio)**: Pick your Topic, Channel, and Format from the dropdowns and tap Play!
   - Generates complete, continuous Telugu narration with **zero silence gaps**.
   - Renders 1080p full HD video with dynamic memes & visuals.
   - Automatically publishes directly to **YouTube** and **Instagram Reels**!
   - Previews the finished video directly on your phone screen.

---


In [ ]:
#@title 🛠️ Step 1: Initialize System & Setup Credentials { display-mode: "form" }
#@markdown Run this cell once when opening the notebook.
import os, sys, subprocess, base64, json, time
from pathlib import Path

print("⏳ [1/4] Installing system FFmpeg & media codecs...")
!apt-get update -qq && apt-get install -y -qq ffmpeg libsm6 libxext6 libgl1 fonts-dejavu > /dev/null

print("⏳ [2/4] Cloning latest AutoTube AI from GitHub...")
if not os.path.exists("/content/autotube-ai"):
    !git clone --depth 1 https://github.com/Harsha79819/autotube-ai.git /content/autotube-ai
else:
    %cd /content/autotube-ai
    !git pull origin main

%cd /content/autotube-ai

print("⏳ [3/4] Installing Python packages (Edge-TTS, MoviePy, Google GenAI, etc.)...")
!pip install -q -r requirements.txt

print("⏳ [4/4] Verifying credentials & channels...")
# Check for secrets from Google Drive or Colab Secrets
try:
    from google.colab import drive
    if os.path.exists("/content/drive/MyDrive/autotube_secrets.env"):
        !cp /content/drive/MyDrive/autotube_secrets.env /content/autotube-ai/.env
        print("✅ Loaded credentials from Google Drive!")
except Exception:
    pass

print("
🚀 AutoTube AI is 100% READY! Scroll to Step 2 to generate videos.")


In [ ]:
#@title 🎬 Step 2: 1-Click Mobile Video Studio (Produce & Upload) { display-mode: "form" }
#@markdown ### 🎯 Select Video Settings:
topic_mode = "Auto-Pick Trending Topic" #@param ["Auto-Pick Trending Topic", "Bigg Boss 10 Today Highlights & Drama", "Tollywood Cinema News & Review", "Mind-Blowing Space & Science Facts", "Tech Innovation & AI Breakthrough", "Custom Topic"]
custom_topic_text = "" #@param {type:"string"}
target_channel = "Channel 2 (viral_talks_me&mine)" #@param ["Channel 2 (viral_talks_me&mine)", "Channel 1 (Imaginary_facts)"]
video_format = "Shorts (9:16 - 60-90s)" #@param ["Shorts (9:16 - 60-90s)", "Long-form (16:9 - 3-5m)"]
voice_anchor = "👩 Shruti (Conversational Telugu Anchor)" #@param ["👩 Shruti (Conversational Telugu Anchor)", "👨 Mohan (Energetic Telugu Anchor)"]
upload_to_youtube = True #@param {type:"boolean"}
upload_to_instagram = True #@param {type:"boolean"}

import os, sys, time, json
from IPython.display import display, HTML, Video

%cd /content/autotube-ai

# Determine channel key
ch_key = "viral_talks_me&mine" if "viral" in target_channel.lower() else "Imaginary_facts"
dur = "3-5m" if "long" in video_format.lower() else "60-90s"
aspect = "16:9" if "long" in video_format.lower() else "9:16"
voice_name = "te-IN-ShrutiNeural" if "shruti" in voice_anchor.lower() else "te-IN-MohanNeural"

# Determine final topic
if topic_mode == "Custom Topic" and custom_topic_text.strip():
    topic = custom_topic_text.strip()
elif topic_mode == "Bigg Boss 10 Today Highlights & Drama":
    topic = "Bigg Boss 10 Telugu Today Nominations and Housemate Arguments"
elif topic_mode == "Tollywood Cinema News & Review":
    topic = "Tollywood Latest Box Office and Trending Movie Updates"
elif topic_mode == "Mind-Blowing Space & Science Facts":
    topic = "Mind-Blowing Space Mysteries and Universe Facts in Telugu"
elif topic_mode == "Tech Innovation & AI Breakthrough":
    topic = "Latest AI Breakthroughs and Smartphone Tech Innovations"
else:
    # Auto-pick trending topic
    from agents.topic_suggester import get_trending_topics
    cat = "entertainment" if ch_key == "viral_talks_me&mine" else "tech"
    suggestions = get_trending_topics(cat, limit=3)
    topic = suggestions[0] if suggestions else "Trending Viral Telugu News"

print("=" * 60)
print(f"🚀 Starting AutoTube Production")
print(f"📌 Topic     : {topic}")
print(f"📺 Channel   : {ch_key}")
print(f"⏱️  Format    : {dur} ({aspect})")
print(f"🎙️ Voice     : {voice_anchor}")
print(f"🌐 YouTube   : {'YES' if upload_to_youtube else 'NO (Render Only)'}")
print(f"📸 Instagram : {'YES' if upload_to_instagram else 'NO'}")
print("=" * 60 + "
")

from pipeline import generate_multi_media_video
res = generate_multi_media_video(
    topic=topic,
    channel=ch_key,
    target_duration=dur,
    voice=voice_name,
    upload_youtube=upload_to_youtube,
    upload_instagram=upload_to_instagram
)

print("
" + "=" * 60)
if res.get("status") == "success":
    video_path = res.get("video_path")
    yt_id = res.get("youtube_id")
    ig_id = res.get("instagram_media_id")
    print("🎉 PRODUCTION & PUBLISHING COMPLETED SUCCESSFULLY!")
    if yt_id:
        print(f"📺 YouTube Live Link : https://youtu.be/{yt_id}")
    if ig_id:
        print(f"📸 Instagram Reel ID  : {ig_id} (@name_less_topics)")
    print("=" * 60)
    if video_path and os.path.exists(video_path):
        print("
🎬 Video Preview:")
        display(Video(video_path, embed=True, width=360))
else:
    print("❌ Execution ended with note/error:", res.get("error"))
print("=" * 60)


In [ ]:
#@title ⏰ Step 3: Run Today's Next Scheduled Peak Slot { display-mode: "form" }
#@markdown Automatically runs whichever slot is due right now (e.g. 08:00 Morning News, 12:30 Tech Explainer, etc.)
%cd /content/autotube-ai
!python daily_scheduler.py --run-current


In [ ]:
#@title 📊 Step 4: 24/7 Channel Health & Quota Dashboard { display-mode: "form" }
#@markdown Check live subscriber counts, views, upload limits, and OAuth token validity for both channels.
%cd /content/autotube-ai
!python analytics/channel_monitor.py
